# ML-02 â€” Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/12-kartik66/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** â€” each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane and why

**Lane 2: Refresh / Content Opportunity Scoring.**

I chose this lane because the starter pipeline already shows a clear gap between a hand-tuned baseline (Precision@50 of 0.240) and a learned model (0.740 for random forest) â€” meaning there is real signal the rules miss and editors could act on. The core decision â€” "which page should someone review first?" â€” is a concrete, capacity-constrained problem with a measurable output (a ranked queue). The starter dataset has 30,000 rows with enough variation in impressions, position, content age, and trend to make the lane worth exploring before moving to the full 79M-row warehouse.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one â€” typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

**What decision does this improve?**

Which pages in a content inventory should an editor review first for refresh, expansion, or monitoring?

**Who acts on the output, and what do they do?**

A content editor or SEO manager receives a ranked queue of pages with scores and reason codes. They inspect the top 20â€“50 candidates and decide whether to refresh (update content), rewrite (improve CTR/engagement), merge (consolidate cannibalising pages), or monitor (watch for further decline).

**What does a wrong recommendation cost?**

- **False positive** (recommending a page that is fine): the editor wastes 10â€“30 minutes reviewing a page that did not need attention. Over 50 recommendations, 20 false positives waste roughly 3â€“7 hours of editorial capacity.
- **False negative** (missing a page that is truly declining and has demand): the page continues losing traffic, costing search visibility, clicks, and potential conversions. The opportunity cost depends on the page's demand volume â€” a page with 10,000 impressions lost could be worth substantially more than editorial time.

**Why does data or ML help at all?**

A plain rule (like the starter baseline of visibility + freshness + position + depth scores) catches obvious cases but misses pages where decline comes from complex, interacting signals â€” a page with high impressions but steeply dropping clicks, or a page holding position while CTR erodes. ML handles many weak signals together and can generalise across clients with different traffic patterns, which a single fixed formula cannot.

**Task type:** Ranking/scoring â€” producing a priority score per page, evaluated by Precision@K (matching how editors actually use the queue).

**Target:** A future-observed outcome â€” for example, whether a page's impressions or clicks decline in the next 30 days relative to its prior window, measured after a clean feature/target split.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one â€” typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter dataset that make Lane 2 worth the next 7 weeks:

In [3]:
import pandas as pd

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# Number 1: over half the content is declining
declining_count = (df['trend_direction'] == 'down').sum()
declining_rate = declining_count / len(df)
print(f'Declining pages: {declining_count} / {len(df)} ({declining_rate:.1%})')
print('---')

# Number 2: the baseline vs ML gap is large
# (From outputs/model_report.md â€” reproduced here for reference)
print('Precision@50 comparison (from the starter pipeline):')
print('  Baseline rules:          0.240 (12/50 correct)')
print('  Random forest:           0.740 (37/50 correct)')
print('  â†’ A learned model found ~25 more actionable pages per 50.')
print('---')

# Number 3: many visible pages are underperforming
# Pages with >500 impressions but very low CTR
high_impressions = df['impressions_90d'] >= 500
low_ctr = df['ctr'] < 0.5  # 0.5% CTR
mask = high_impressions & low_ctr & (df['avg_position'] > 0) & (df['avg_position'] <= 20)
print(f'Pages with >=500 impressions, CTR <0.5%, position 1-20: {mask.sum():,}')
print(f'  These are visible pages under-capturing clicks â€” prime refresh candidates.')
print('---')

# Bonus: total impressions at stake
print(f'Total impressions across dataset: {df["impressions_90d"].sum():,}')
print(f'Total clicks:                     {df["clicks_90d"].sum():,}')
print(f'  Even small improvements at scale mean real traffic gains.')

Declining pages: 16262 / 30000 (54.2%)
---
Precision@50 comparison (from the starter pipeline):
  Baseline rules:          0.240 (12/50 correct)
  Random forest:           0.740 (37/50 correct)
  â†’ A learned model found ~25 more actionable pages per 50.
---
Pages with >=500 impressions, CTR <0.5%, position 1-20: 9,759
  These are visible pages under-capturing clicks â€” prime refresh candidates.
---
Total impressions across dataset: 156,010,989
Total clicks:                     482,920
  Even small improvements at scale mean real traffic gains.


## 4. Careful words: what I can and can't claim

**What I can claim (decision-support, observed, directional):**

- That a ranked queue built from observable signals can prioritise pages differently from a simple rule.
- That Precision@K on a held-out set measures how well the ranking matches a future-observed outcome.
- Observed associations between signals (position, impressions, content age, trend) and future decline â€” with effect sizes, not just p-values.
- A comparison of model performance to a transparent baseline, with honest confidence intervals.

**What I cannot claim:**

- That refreshing a recommended page caused a recovery â€” this is observational work, not a causal experiment.
- That the model predicts Google's algorithm or rankings â€” it uses search and engagement signals, not algorithm factors.
- That a page ranked low is "safe" â€” low rank simply means no evidence of near-term opportunity was found.
- Any result that generalises beyond the dataset's date window or client set without explicit replication.

**The target must be future-observed.** The starter's `is_declining_label` is a current-window proxy. For the capstone, the label will be defined from a clean time split: features from a prior window (e.g. days 1â€“90) predicting an outcome in a later window (e.g. days 91â€“120). The data contract in week 03 will spell out the exact windows and the leakage audit.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one â€” typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled â€” markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime â†’ Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` â€” then submit your repo URL on the card. Done.